In [1]:
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt

In [2]:
# Saving all .csv files in folder to list.
#path = "MachineLearningCVE/"
#files = [file for file in glob.glob(path + "**/*.csv", recursive=True)]

In [3]:
#[print(f) for f in files]

In [4]:
# Reading all the csv files into dataframes and putting thoose DFs to one list.

#dataset = [pd.read_csv(f) for f in files]

In [5]:
# shape of the each files

#for d in dataset:
    #print(d.shape)


In [6]:
# comparing the columns of each tables

#for i in range(0,len(dataset)):
 #   if i != len(dataset)-1:
    #    same_columns = dataset[i].columns == dataset[i+1].columns
        
      #  if False in same_columns:
        #    print(i)
      #      break

   # same_columns

In [7]:
# Combining all tables into one dataset

#dataset = pd.concat([d for d in dataset]).drop_duplicates(keep=False)
#dataset.reset_index(drop=True, inplace = True)

In [8]:
# Saving combined data

#dataset.to_csv("CICIDS2017/Dataset_work.csv", index=False)

In [9]:
dataset = pd.read_csv("Dataset_work.csv")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x80 in position 32826: invalid start byte

In [ ]:
# dataset shape

dataset.shape


In [ ]:
# checking the datatypes of each features
#dataset.dtypes
dataset.info()

In [ ]:
# descriptive statistics of the dataset
dataset.describe()
#dataset.describe().transpose()

In [ ]:
# actual dataset

dataset.head(10)

In [ ]:
# Removing whitespaces in column names.

col_names = [col.replace(' ', '') for col in dataset.columns]
dataset.columns = col_names
dataset.head()

In [ ]:
# Dataset conatains 15 labels.
#print(dataset[' Label'].unique())
#len(dataset[' Label'].unique())

dataset['Label'].value_counts()

In [ ]:
# This next snippet uses regular expressions to replace wierd characters with dunders.

label_names = dataset['Label'].unique()


import re

label_names = [re.sub("[^a-zA-Z ]+", "", l) for l in label_names]
label_names = [re.sub("[\s\s]", '_', l) for l in label_names]
label_names = [lab.replace("__", "_") for lab in label_names]

label_names, len(label_names)

In [ ]:
# Replacing 'Label' column values with new readable values.

labels = dataset['Label'].unique()

for i in range(0,len(label_names)):
    dataset['Label'] = dataset['Label'].replace({labels[i] : label_names[i]})
    
dataset['Label'].unique()

In [ ]:
# changing attack labels to their respective attack class
# total 38 different sub-attacks 
def change_label(dataset):
    dataset.Label.replace(['DoS_slowloris','DoS_Slowhttptest','DoS_Hulk','DoS_GoldenEye','DDoS'],'Dos',inplace=True)
    dataset.Label.replace(['Web_Attack_Brute_Force','Web_Attack_XSS', 'Web_Attack_Sql_Injection','Bot'],'Botnet',inplace=True)
    dataset.Label.replace(['Heartbleed','PortScan'],'PortScan',inplace=True)
    dataset.Label.replace(['Infiltration','FTPPatator','SSHPatator'],'Firewall',inplace=True)


In [ ]:
# calling change_label() function
change_label(dataset)

In [ ]:
# distribution of attack classes
dataset.Label.value_counts()

In [ ]:
# checking null values 

dataset.isnull().values.any()

In [ ]:
 #Checking which column/s contain NULL values.

[col for col in dataset if dataset[col].isnull().values.any()]


In [ ]:
# Checking how many NULL values it this column contains.

dataset['FlowBytes/s'].isnull().sum()

In [ ]:
# Removing rows that contain NULL values

dataset.dropna(inplace=True)

In [ ]:
dataset.isnull().any().any()

In [ ]:
# converting string values to float64 

labl = dataset['Label']
dataset = dataset.loc[:, dataset.columns != 'Label'].astype('float64')

In [ ]:


# Checking if all values are finite.

np.all(np.isfinite(dataset))



In [ ]:


# Checking what column/s contain non-finite values.

nonfinite = [col for col in dataset if not np.all(np.isfinite(dataset[col]))]

nonfinite



In [ ]:
# Checking how many non-finite values each column contains.

finite = np.isfinite(dataset['FlowBytes/s']).sum()

dataset.shape[0] - finite

In [ ]:
# Replacing infinite values with NaN values.
dataset = dataset.replace([np.inf, -np.inf], np.nan)

In [ ]:
np.any(np.isnan(dataset))

In [ ]:
dataset = dataset.merge(labl, how='outer', left_index=True, right_index=True)

In [ ]:
dataset.dropna(inplace=True)

In [ ]:
dataset.shape

In [ ]:
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler

In [ ]:
# selecting numeric attributes columns from data
numeric_col = dataset.select_dtypes(include='number').columns

# using standard scaler for normalizing
std_scaler = StandardScaler()
def normalization(df,col):
  for i in col:
    arr = df[i]
    arr = np.array(arr)
    df[i] = std_scaler.fit_transform(arr.reshape(len(arr),1))
  return df


In [ ]:
# calling the normalization() function
dataset = normalization(dataset.copy(),numeric_col)

# data after normalization
dataset.head()

In [ ]:
# multiclass classification ##

In [ ]:
multi_data = dataset.copy()
multi_label = pd.DataFrame(multi_data.Label)

In [ ]:
from sklearn.preprocessing import LabelEncoder
LE2 = LabelEncoder()

enc_label = multi_label.apply(LE2.fit_transform)
multi_data['intrusion']= enc_label

In [ ]:
LE2.classes_

In [ ]:
multi_data.head()

In [ ]:
# one-hot-encoding for attack label

multi_data = pd.get_dummies(multi_data,columns=['Label'],prefix="",prefix_sep="")
multi_data['Label']= multi_label
multi_data

In [ ]:
# attack distribution for multiclass

plt.figure(figsize=(10,10))
plt.pie(multi_data.Label.value_counts(),labels=multi_data.Label.unique(),autopct='%0.2f%%')
plt.title("pie chart distribution of multi-class labels")
plt.legend()
plt.savefig('CICIDS2017/Pie_chart_multi.png')
plt.show()

In [ ]:
## feature extraction ##

In [ ]:
## feature extraction for multi class classification ##

In [ ]:
numeric_multi = multi_data[numeric_col]
numeric_multi['intrusion'] = multi_data['intrusion']

In [ ]:
corr=numeric_multi.corr()
corr_y = abs(corr['intrusion'])
highest_corr = corr_y[corr_y >0.2]
highest_corr.sort_values(ascending=True)

In [ ]:
numeric_multi = multi_data[['FINFlagCount','FlowDuration','FwdIATTotal','BwdPacketLengthMin','MinPacketLength','AveragePacketSize','PacketLengthMean',
                           'PacketLengthVariance','FlowIATStd','MaxPacketLength','PacketLengthStd','AvgBwdSegmentSize','IdleMin',
                           'BwdPacketLengthMean','FlowIATMax','BwdPacketLengthMax','IdleMax','BwdPacketLengthStd','FwdIATMax',
                           'FwdIATStd','IdleMean']]

In [ ]:

multi_data = numeric_multi.join(multi_data[['intrusion','BENIGN','Botnet', 'Dos', 'Firewall', 'PortScan','Label']])

In [ ]:
multi_data.to_csv("CICIDS2017/multi_data_balanced.csv")
multi_data

In [ ]:
# counting the attacks for multiclass

multi_data['intrusion'].value_counts()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score,auc,roc_curve,precision_recall_fscore_support
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
import time
import warnings

In [ ]:
 # classifier for multi-classification dataset

warnings.filterwarnings("ignore")

X = multi_data.iloc[:,0:21].to_numpy() # dataset excluding target attribute (encoded, one-hot-encoded,original)
y = multi_data['intrusion'] # target attribute

X_train, X_validation, Y_train, Y_validation = train_test_split(X, y, test_size=0.20, random_state=1)
# Spot Check Algorithms
models = []
models.append(('LR', LogisticRegression(solver='lbfgs', multi_class='ovr')))
models.append(('LDA', LinearDiscriminantAnalysis(n_components=3)))
models.append(('KNN', KNeighborsClassifier(n_neighbors=3)))
models.append(('CART', DecisionTreeClassifier(max_depth=5)))
models.append(('SVM', SVC(gamma=10,max_iter=300)))

print ('Model\tAcc\tFAR\tsen\tspe\tExecution_time')

# evaluate each model in turn
results = []
names = []

for name, model in models:
    start_time = time.time()
    kfold = StratifiedKFold(n_splits=10, random_state=1, shuffle=True)
    
    cv_results = cross_val_score(model, X_train, Y_train, cv=kfold, scoring='accuracy').mean()
    
    
    m = model.fit(X_train, Y_train)
    predict = m.predict(X_validation)
    delta = time.time()- start_time
    cm = confusion_matrix(Y_validation, predict)
    # Creating a dataframe for a array-formatted Confusion matrix,so it will be easy for plotting.
    cm_df = pd.DataFrame(cm)
    
    total1=sum(sum(cm))
    false_alaram_rate = cm[1,0]/(cm[1,0]+cm[0,0])
    sensitivity = cm[1,1]/(cm[1,1]+cm[0,1])
    specificity = cm[0,0]/(cm[1,0]+cm[0,0])
    f1score = precision_recall_fscore_support(Y_validation, predict,average='weighted')
    
   
    results.append(cv_results)
    names.append(name)
    print('{}\t{:.3f}\t{:.3f}\t{:.3f}\t{:.3f}\t{:.2f} sec'.format(name, cv_results, false_alaram_rate, sensitivity, specificity, delta))
    print(f1score)
    print(cm_df)

In [ ]:
 #roc auc plot for multiclass
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from sklearn.multiclass import OneVsRestClassifier
from itertools import cycle

X = multi_data.iloc[:,0:16].to_numpy() # dataset excluding target attribute (encoded, one-hot-encoded,original)
y = multi_data['intrusion'] # target attribute

# Binarize the output
y = label_binarize(y, classes=[0, 1, 2,3,4])
n_classes = y.shape[1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=0)

classifier = OneVsRestClassifier(KNeighborsClassifier(n_neighbors=3))
y_score = classifier.fit(X_train, y_train)
ypred = y_score.predict(X_test)

fpr = dict()
tpr = dict()
roc_auc = dict()
lw=2
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test[:, i], ypred[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label='ROC curve of class {0} (area = {1:0.2f})'
             ''.format(i, roc_auc[i]))
plt.plot([0, 1], [0, 1], 'k--', lw=lw)
plt.xlim([-0.05, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver operating characteristic for multi-class data (KNN)')
plt.legend(loc="lower right")
plt.show()